# Part 3: Classical ML Baselines
## Random Forest, SVM, XGBoost on Molecular Fingerprints
 
Establishes baseline performance using traditional ML models on:
- Morgan Fingerprints (2048-bit)
- MACCS Keys (166-bit)
- Morgan + MACCS (2214-bit)

In [ ]:
# @title 1. Setup
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load preprocessed data
train_df = pd.read_csv('data/train.csv')
val_df = pd.read_csv('data/val.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
# @title 2. Extract Fingerprints
from vegfr2.features import smiles_to_morgan, smiles_to_maccs

def extract_features(df):
    morgan = np.vstack([smiles_to_morgan(s) for s in df['smiles']])
    maccs = np.vstack([smiles_to_maccs(s) for s in df['smiles']])
    both = np.hstack([morgan, maccs])
    return morgan, maccs, both

X_train_morgan, X_train_maccs, X_train_both = extract_features(train_df)
X_val_morgan, X_val_maccs, X_val_both = extract_features(val_df)
X_test_morgan, X_test_maccs, X_test_both = extract_features(test_df)

y_train = train_df['active'].values
y_val = val_df['active'].values
y_test = test_df['active'].values

print(f"Morgan: {X_train_morgan.shape[1]}-dim")
print(f"MACCS: {X_train_maccs.shape[1]}-dim")
print(f"Both: {X_train_both.shape[1]}-dim")

In [ ]:
# @title 3. Train Models
from vegfr2.ml_models import train_ml_model, predict_ml_model
from vegfr2.metrics import classification_metrics

results = {}
models = {}

feature_sets = {
    'Morgan': (X_train_morgan, X_test_morgan),
    'MACCS': (X_train_maccs, X_test_maccs),
    'Morgan+MACCS': (X_train_both, X_test_both),
}

for model_name in ['rf', 'svm', 'xgb']:
    for fp_name, (X_tr, X_te) in feature_sets.items():
        name = f"{model_name}_{fp_name.lower().replace('+', '_')}"
        print(f"Training {name}...")
        
        model = train_ml_model(model_name, X_tr, y_train, seed=42)
        probs = predict_ml_model(model, X_te)
        metrics = classification_metrics(y_test.tolist(), probs.tolist())
        
        results[name] = metrics
        models[name] = model
        
        print(f"  AUC={metrics.get('auc', 0):.4f} ACC={metrics['acc']:.4f} MCC={metrics['mcc']:.4f}")

In [ ]:
# @title 4. Results Table
print("=" * 70)
print("CLASSICAL ML RESULTS")
print("=" * 70)

header = f"{'Model':<30} {'ACC':>6} {'SEN':>6} {'SPE':>6} {'MCC':>6} {'AUC':>6}"
print(header)
print("-" * 70)

for name, m in sorted(results.items(), key=lambda x: x[1].get('auc') or 0, reverse=True):
    auc_str = f"{m['auc']:.4f}" if m.get('auc') is not None else "N/A"
    print(f"{name:<30} {m['acc']:.4f} {m['sen']:.4f} {m['spe']:.4f} {m['mcc']:.4f} {auc_str:>6}")

In [ ]:
# @title 5. ROC Curves
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (fp_name, (_, X_te)) in enumerate(feature_sets.items()):
    ax = axes[idx]
    for model_name in ['rf', 'svm', 'xgb']:
        name = f"{model_name}_{fp_name.lower().replace('+', '_')}"
        probs = predict_ml_model(models[name], X_te)
        fpr, tpr, _ = roc_curve(y_test, probs)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{model_name.upper()} (AUC={roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve - {fp_name}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/roc_curves_ml.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# @title 6. Feature Importance (Random Forest)
from sklearn.ensemble import RandomForestClassifier

# Train RF on Morgan+MACCS for feature importance
rf_full = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_full.fit(X_train_both, y_train)

# Morgan importance
importance_morgan = rf_full.feature_importances_[:2048]
importance_maccs = rf_full.feature_importances_[2048:]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(2048), importance_morgan, width=1.0, alpha=0.7)
axes[0].set_title('Morgan Fingerprint Feature Importance')
axes[0].set_xlabel('Bit Index')
axes[0].set_ylabel('Importance')

axes[1].bar(range(166), importance_maccs, width=1.0, alpha=0.7, color='orange')
axes[1].set_title('MACCS Key Feature Importance')
axes[1].set_xlabel('Key Index')
axes[1].set_ylabel('Importance')

plt.tight_layout()
plt.savefig('images/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Top 10 most important Morgan bits: {np.argsort(importance_morgan)[-10:][::-1]}")
print(f"Top 10 most important MACCS keys: {np.argsort(importance_maccs)[-10:][::-1]}")

In [ ]:
# @title 7. Save Models
import pickle

for name, model in models.items():
    path = Path(f'models/{name}.pkl')
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(model, f)

print(f"Saved {len(models)} models to models/")

# Save results
results_df = pd.DataFrame([
    {'model': name, **m} for name, m in results.items()
])
results_df.to_csv('data/ml_results.csv', index=False)
print("Results saved to data/ml_results.csv")